In [ ]:
pip install openai python-dotenv pandas tqdm

In [ ]:
import os
import json
import re
import pandas as pd
from pathlib import Path
from tqdm import tqdm
from getpass import getpass
from concurrent.futures import ThreadPoolExecutor, as_completed
from openai import OpenAI

project_root = Path(r"C:\Users\Luiza\OneDrive\Desktop\MA THESIS\FINAL\FINAL_NARR_prompting")

raw_text_root = project_root / "raw_texts"
llm_output_root = project_root / "llm_outputs"
llm_output_root.mkdir(exist_ok=True)

final_teds = ["TED_001", "TED_002", "TED_004", "TED_005", "TED_006", "TED_007", "TED_008", "TED_010", "TED_011", "TED_012", "TED_015", "TED_016", "TED_017", "TED_018", "TED_019", "TED_020"]
#optimization teds = ["TED_003", "TED_009", "TED_013", "TED_14"]
def read_ted_text(document_id):
    with open(raw_text_root / f"{document_id}.txt", "r", encoding="utf-8") as f:
        return f.read()

# set up

In [ ]:
client = OpenAI(
    api_key=# API KEY HERE,
)

#  models

In [ ]:
models = [
    "mistralai/mistral-small-3.2-24b-instruct",
    "qwen/qwen3.6-27b", 
    "google/gemma-4-31b-it"
]

model_file_names = {
    "mistralai/mistral-small-3.2-24b-instruct": "mistral-small-3.2-24b-instruct",
    "qwen/qwen3.6-27b": "qwen3.6-27b",
    "google/gemma-4-31b-it": "gemma-4-31b-it"
}


# read ted text

In [ ]:
def read_ted_text(document_id):
    path = raw_text_root / f"{document_id}.txt"

    with open(path, "r", encoding="utf-8") as f:
        return f.read()


# FINAL NARRATION PROMPT: used on the other 16 ted talks, and the examples used are from developmental teds

In [ ]:
system_prompt_narrative_final = """
You are an expert annotator trained in narrative discourse analysis and argumentation mining.

Your task is to identify narrative spans in TED Talk transcripts and classify their rhetorical function within the broader argument.

Return ONLY valid JSON.
Do not explain your reasoning.
Do not use markdown.
"""

def build_narrative_prompt_final(ted_text):
    return f"""
TASK:
Annotate narrative spans in the TED Talk.

A narrative span is a segment of text that includes story-like elements such as:
- events,
- actions,
- situations,
- people or characters,
- temporal progression,
- causal progression,
- descriptions of what happened,
- examples or anecdotes.

Only annotate narrative spans that contribute to the discourse.

Use ONLY these labels based on their definitions:

1. Exordium
A narrative opening that engages the audience, introduces the topic, or draws attention.
Usually appears near the beginning, but label by function, not position.

2. Narratio
Narrative background or context that helps the audience understand the issue.
Use this for situations, historical background, prior events, or conditions that explain why the topic matters.

3. Propositio
A narrative span that introduces or frames the speaker's central stance, project, purpose, or direction of the argument.
Use this when the narrative helps present what the speaker is arguing for or what the talk will develop.

4. Confirmatio
A narrative span that functions as evidence, support, illustration, example, case study, or proof for the speaker's argument. In some cases, narrative evidence may also implicitly challenge alternative viewpoints by presenting contrasting outcomes
Use this for concrete events or examples that support a claim.

6. Peroratio
A narrative closing that summarizes, reinforces, or leaves a final impression supporting the overall argument.
Usually appears near the end, but label by function, not position.

7. Other
A narrative span that contributes to the discourse but does not clearly fit the labels above.
Use this sparingly.

IMPORTANT DECISION RULES:

- Label by rhetorical function, not by location.
- A span is not NARRATIO only because it appears early.
- A span is not PERORATIO only because it appears at the end.
- A personal anecdote is not automatically argumentative.
- A description is not automatically narrative evidence.
- Only annotate spans that have a clear narrative function in the talk.
- If a passage is purely argumentative with no narrative elements, do not annotate it.
- If a passage is only a greeting, joke, transition, or filler, do not annotate it.
- If unsure, do not annotate.

SPAN RULES:

- Copy exact text spans from the TED Talk.
- Do not paraphrase.
- Do not invent spans.
- Select complete narrative units, not isolated words.
- Prefer one coherent narrative segment over many tiny fragments.
- Do not include unnecessary surrounding text.
- Do not duplicate the same span with multiple labels.
- Choose the single best functional label.

IMPORTANT
- Most spans have more than one sentence, to which only one single label is assigned.
- A text doesn't necessarily have to include all of the narrative labels.


TED TALK:
\"\"\"
{ted_text}
\"\"\"
"""


# clean JSON output

In [ ]:
def clean_llm_json_output(output):
    output = output.strip()

    output = re.sub(r"```json", "", output, flags=re.IGNORECASE)
    output = re.sub(r"```", "", output)

    start = output.find("{")
    end = output.rfind("}")

    if start == -1 or end == -1 or end <= start:
        raise ValueError("No JSON object found in model output.")

    output = output[start:end+1]

    parsed = json.loads(output)

    return json.dumps(parsed, ensure_ascii=False, indent=2)

# call openrouter

In [ ]:
def run_openrouter_prompt(model, system_prompt, user_prompt, temperature=0):
    kwargs = {
        "model": model,
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ],
        "temperature": temperature
    }

    # Sara said this may help for Gemma 4
    if model == "google/gemma-4-31b-it":
        kwargs["extra_body"] = {"reasoning": {"enabled": True}}

    completion = client.chat.completions.create(**kwargs)
    return completion.choices[0].message.content

# run all MODELS + all 16 TED TALKS

In [ ]:
prompt_version = "final"          
task_type = "narration"

def run_one_job(document_id, model):
    ted_text = read_ted_text(document_id)

    safe_model_name = model_file_names[model]

    output_filename = (
        f"{document_id}_{safe_model_name}_"
        f"prompt-{prompt_version}_{task_type}.json"
    )

    output_path = llm_output_root / output_filename
    raw_error_path = llm_output_root / output_filename.replace(".json", "_RAW_ERROR.txt")

    raw_output = run_openrouter_prompt(
        model=model,
        system_prompt=system_prompt_narrative_final,
        user_prompt=build_narrative_prompt_final(ted_text),
        temperature=0
    )

    try:
        cleaned_output = clean_llm_json_output(raw_output)

        with open(output_path, "w", encoding="utf-8") as f:
            f.write(cleaned_output)

        status = "success"

    except Exception as e:
        with open(raw_error_path, "w", encoding="utf-8") as f:
            f.write(raw_output)

        status = f"json_cleaning_error: {e}"

    return {
        "document": document_id,
        "model": safe_model_name,
        "prompt_version": prompt_version,
        "task_type": task_type,
        "output_file": str(output_path),
        "status": status
    }


jobs = []

for document_id in final_teds:
    for model in models:
        jobs.append((document_id, model))

run_log = []

with ThreadPoolExecutor(max_workers=1) as executor:
    futures = [
        executor.submit(run_one_job, document_id, model)
        for document_id, model in jobs
    ]

    for future in tqdm(as_completed(futures), total=len(futures)):
        try:
            run_log.append(future.result())
        except Exception as e:
            run_log.append({
                "document": None,
                "model": None,
                "prompt_version": prompt_version,
                "task_type": task_type,
                "output_file": None,
                "status": f"api_error: {e}"
            })

run_log_df = pd.DataFrame(run_log)
run_log_df

# Check the status errors

In [ ]:
failed = run_log_df[
    run_log_df["status"].str.contains("error", case=False, na=False)
]

for i, row in failed.iterrows():
    print("\n====================")
    print("MODEL:", row["model"])
    print("STATUS:")
    print(row["status"])

# converting RAW_ERROR (.txt) files into JSON

In [ ]:
import json
import re
from pathlib import Path

# Folder containing your RAW_ERROR .txt files
input_folder = Path(r"C:\Users\Luiza\OneDrive\Desktop\MA THESIS\FINAL\FINAL_NARR_prompting\llm_outputs")

# Folder where converted JSON files will be saved
output_folder = Path(r"C:\Users\Luiza\OneDrive\Desktop\MA THESIS\FINAL\FINAL_NARR_prompting\fixed_llm_output")
output_folder.mkdir(parents=True, exist_ok=True)

def load_raw_jsonish_txt(path):
    raw = path.read_text(encoding="utf-8").strip()

    # Remove ```json ... ``` wrappers if present
    raw = re.sub(r"^```(?:json)?\s*", "", raw)
    raw = re.sub(r"\s*```$", "", raw)

    return json.loads(raw)

def convert_file(path):
    data = load_raw_jsonish_txt(path)

    annotations = []
    for item in data:
        text_value = item.get("span") or item.get("text")

        if text_value is None:
            print(f"Skipped item without span/text in {path.name}")
            continue

        annotations.append({
            "label": item.get("label", "Narratio"),
            "text": text_value
        })

    return {"annotations": annotations}

for txt_path in input_folder.glob("*.txt"):
    converted = convert_file(txt_path)

    output_name = txt_path.stem.replace("_RAW_ERROR", "") + ".json"
    output_path = output_folder / output_name

    with output_path.open("w", encoding="utf-8") as f:
        json.dump(converted, f, ensure_ascii=False, indent=2)

    print(f"Saved: {output_path}")

In [ ]:
run_log_path = llm_output_root / f"run_log_prompt-{prompt_version}_{task_type}.csv"

run_log_df.to_csv(run_log_path, index=False)

print("Saved run log to:", run_log_path)

In [ ]:
# async def run_single(document_id, model_short_name, model_id):
#     ted_text = read_ted_text(document_id)

#     output_filename = (
#         f"{document_id}_{model_short_name}_"
#         f"prompt-{prompt_version}_argumentation.json"
#     )

#     output_path = llm_output_root / output_filename

#     raw_error_path = llm_output_root / output_filename.replace(
#         ".json",
#         "_RAW_ERROR.txt"
#     )

#     try:
#         raw_output = await run_openrouter_prompt(
#             model_id=model_id,
#             system_prompt=system_prompt_argumentation_v8,
#             user_prompt=build_argumentation_prompt_v8(ted_text),
#         )

#         cleaned_output = clean_llm_json_output(raw_output)

#         with open(output_path, "w", encoding="utf-8") as f:
#             f.write(cleaned_output)

#         return {
#             "document": document_id,
#             "model": model_short_name,
#             "model_id": model_id,
#             "prompt_version": prompt_version,
#             "task_type": "argumentation",
#             "output_file": str(output_path),
#             "status": "success",
#         }

#     except Exception as e:
#         if "raw_output" in locals():
#             with open(raw_error_path, "w", encoding="utf-8") as f:
#                 f.write(raw_output)

#         return {
#             "document": document_id,
#             "model": model_short_name,
#             "model_id": model_id,
#             "prompt_version": prompt_version,
#             "task_type": "argumentation",
#             "output_file": str(output_path),
#             "status": f"error: {e}",
#         }
